# CHB12 Data Preparation for Time-to-Seizure Forecasting

**Purpose.** Add the seizure-dense CHB12 subject (CHB-MIT Scalp EEG Database)
to the existing CHB24 feature representation, using EXACTLY the same
preprocessing/schema, so the two subjects' feature tables can later be
combined for retraining the State-Deviation-Index -> TCN pipeline.

**This notebook does NOT:**
- train a TCN, CNN/EEGNet, or any model,
- construct the final 4-class time-to-seizure target,
- download the full CHB12 recording set (only seizure-containing, montage-
  compatible EDFs are fetched),
- modify `EEG_preprocessing.ipynb` or `EEG_CHB24_preprocessing.ipynb` or
  their outputs.

**Reuse policy.** All preprocessing logic below (channel selection,
filtering, epoching, Welch-PSD feature extraction, CHB summary parsing,
seizure labeling, temporal metadata) is copied **verbatim** from the
verified cells of `EEG_CHB24_preprocessing.ipynb` (itself copied verbatim
from `EEG_preprocessing.ipynb`), to guarantee CHB12 is processed identically
to CHB01/CHB24. The ONE piece of genuinely new logic -- because CHB12's raw
EDFs are not montage-uniform across the whole subject -- is an explicit,
verified montage classification/exclusion step and a channel-name filter
applied BEFORE the verbatim `preprocess_eeg()` call; both are clearly marked
below.

**Source of truth for parameters.** `outputs/preprocessing_metadata.json`
and an actual loaded CHB24 EDF header (for the exact runtime channel names,
including MNE's automatic duplicate-channel renaming) are used as ground
truth, rather than re-declaring constants or trusting summary-file text
alone.

**Mandatory checkpointing.** Each EDF is downloaded, preprocessed, and
saved as its own checkpoint parquet under
`outputs/preprocessed_data/chb12_checkpoints/` before moving to the next
one; RAM is released (`del` + `gc.collect()`) between recordings. On rerun,
any checkpoint that already exists and validates is skipped, so this
notebook is safe to restart after a VS Code/WSL/kernel crash.

In [1]:
print("=" * 60)
print("CHB12 DATA PREPARATION FOR TIME-TO-SEIZURE FORECASTING")
print("=" * 60)

import gc
import json
import re
from collections import Counter
import urllib.request
import urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import mne
from scipy.signal import welch

print("\nLibrary versions:")
print("  numpy:", np.__version__)
print("  pandas:", pd.__version__)
print("  mne:", mne.__version__)

repo_root = Path("/home/evgenija/projects/EEG_timeseries_project")
data_root = repo_root / "data"
output_dir = repo_root / "outputs"
chb_dir = data_root / "raw" / "chbmit"   # SAME directory as CHB01/CHB24 (existing convention)
chb_dir.mkdir(parents=True, exist_ok=True)

print(f"\nrepo_root : {repo_root}")
print(f"chb_dir   : {chb_dir}  (existing CHB01/CHB24 files also live here)")
print(f"output_dir: {output_dir}")

CHB12 DATA PREPARATION FOR TIME-TO-SEIZURE FORECASTING



Library versions:
  numpy: 2.5.2
  pandas: 3.0.5
  mne: 1.12.1

repo_root : /home/evgenija/projects/EEG_timeseries_project
chb_dir   : /home/evgenija/projects/EEG_timeseries_project/data/raw/chbmit  (existing CHB01/CHB24 files also live here)
output_dir: /home/evgenija/projects/EEG_timeseries_project/outputs


### What did we just do, and why?

Standard imports and paths, identical to `EEG_CHB24_preprocessing.ipynb`.
`chb_dir` is the same shared download directory already used for CHB01 and
CHB24 -- no new data layout is introduced.

**Connection to the next step.** Before touching CHB12, we load the
already-verified CHB24 preprocessing parameters and, critically, the ACTUAL
runtime EEG channel names MNE produces for this dataset family (including
its automatic renaming of duplicate channel labels) -- this is the exact
reference CHB12's channels must be reconciled against.

In [2]:
print("\n" + "=" * 60)
print("STEP 1: INSPECT EXISTING VERIFIED CHB24 PIPELINE (GROUND TRUTH)")
print("=" * 60)

existing_metadata_path = output_dir / "preprocessing_metadata.json"
existing_chb24_parquet_path = output_dir / "features_chb24_forecasting.parquet"

if not existing_metadata_path.exists():
    raise FileNotFoundError(f"{existing_metadata_path} not found -- EEG_preprocessing.ipynb must run first.")
if not existing_chb24_parquet_path.exists():
    raise FileNotFoundError(
        f"{existing_chb24_parquet_path} not found -- EEG_CHB24_preprocessing.ipynb must run first."
    )

with open(existing_metadata_path) as f:
    existing_metadata = json.load(f)

existing_chb24_df = pd.read_parquet(existing_chb24_parquet_path)

FILTER_LOW_HZ = existing_metadata["preprocessing_params"]["filter_low_hz"]
FILTER_HIGH_HZ = existing_metadata["preprocessing_params"]["filter_high_hz"]
EPOCH_DURATION_SEC = existing_metadata["preprocessing_params"]["epoch_duration_sec"]
EPOCH_OVERLAP = existing_metadata["preprocessing_params"]["epoch_overlap"]
expected_chb_sfreq = existing_metadata["sampling_rates_hz"]["chb"]

assert FILTER_LOW_HZ == 0.5 and FILTER_HIGH_HZ == 40.0, "Unexpected CHB filter params"
assert EPOCH_DURATION_SEC == 30.0 and EPOCH_OVERLAP == 0.0, "Unexpected CHB epoch params"

qeeg_feature_cols = [
    "std", "var", "rms", "delta", "theta", "alpha", "beta",
    "rel_delta", "rel_theta", "rel_alpha", "rel_beta",
    "theta_alpha_ratio", "delta_beta_ratio", "entropy",
]
missing_expected = [c for c in qeeg_feature_cols if c not in existing_chb24_df.columns]
if missing_expected:
    raise ValueError(f"Existing CHB24 table missing expected qEEG columns: {missing_expected}")

print(f"Verified params: filter={FILTER_LOW_HZ}-{FILTER_HIGH_HZ}Hz, "
      f"epoch={EPOCH_DURATION_SEC}s/overlap={EPOCH_OVERLAP}, sfreq={expected_chb_sfreq}Hz")
print(f"Existing CHB24 table: shape={existing_chb24_df.shape}, "
      f"recordings={sorted(existing_chb24_df['recording_id'].unique())}")

# NEW (CHB12-specific): load one real, already-downloaded CHB24 EDF header
# to obtain the ACTUAL runtime channel names MNE assigns -- including its
# automatic renaming of duplicate labels (verified: this dataset's montage
# declares "T8-P8" twice, and MNE deterministically renames these in
# encounter order to "T8-P8-0"/"T8-P8-1"; the raw summary-file text alone
# does not reflect this). This, not the summary.txt text, is the canonical
# reference CHB12's own channels are checked against below.
reference_chb24_edf = chb_dir / "chb24_01.edf"
if not reference_chb24_edf.exists():
    raise FileNotFoundError(f"{reference_chb24_edf} not found -- need a verified CHB24 EDF as channel reference")

_ref_raw = mne.io.read_raw_edf(reference_chb24_edf, preload=False, verbose=False)
CANONICAL_EEG_CHANNELS = list(_ref_raw.ch_names)
del _ref_raw

print(f"\n[REFERENCE] Canonical {len(CANONICAL_EEG_CHANNELS)}-channel EEG montage "
      f"(actual MNE runtime names, from {reference_chb24_edf.name}):")
print(f"  {CANONICAL_EEG_CHANNELS}")


STEP 1: INSPECT EXISTING VERIFIED CHB24 PIPELINE (GROUND TRUTH)
Verified params: filter=0.5-40.0Hz, epoch=30.0s/overlap=0.0, sfreq=256.0Hz
Existing CHB24 table: shape=(1440, 24), recordings=['chb_s24_r01', 'chb_s24_r03', 'chb_s24_r04', 'chb_s24_r06', 'chb_s24_r07', 'chb_s24_r09', 'chb_s24_r11', 'chb_s24_r13', 'chb_s24_r14', 'chb_s24_r15', 'chb_s24_r17', 'chb_s24_r21']



[REFERENCE] Canonical 23-channel EEG montage (actual MNE runtime names, from chb24_01.edf):
  ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']


/tmp/ipykernel_53220/2175212543.py:54: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  _ref_raw = mne.io.read_raw_edf(reference_chb24_edf, preload=False, verbose=False)


### What did we just do, and why?

We loaded the verified CHB24 preprocessing parameters and feature table as
ground truth (identical logic to `EEG_CHB24_preprocessing.ipynb` Step 1),
and additionally opened one already-verified CHB24 EDF header to capture
`CANONICAL_EEG_CHANNELS` -- the exact 23 channel names MNE actually produces
at runtime for this project's established CHB montage, duplicate-renaming
included. Comparing against this (rather than against the summary.txt text)
is what makes the montage-compatibility check below trustworthy: it reflects
what the code will really see, not what the documentation claims.

**Connection to the next step.** Download and parse `chb12-summary.txt`,
which is CHB12's own source of truth for file list, seizure timing, AND
(critically for CHB12, unlike CHB01/CHB24) which channel montage applies to
each file -- CHB12's montage changes partway through the subject.

In [3]:
print("\n" + "=" * 60)
print("STEP 2: DOWNLOAD + PARSE chb12-summary.txt (SOURCE OF TRUTH)")
print("=" * 60)

CHB12_BASE_URL = "https://physionet.org/files/chbmit/1.0.0/chb12/"
chb12_summary_path = chb_dir / "chb12-summary.txt"

def download_if_missing(url, dest_path, min_bytes=1000):
    '''Idempotent download: skip if dest_path already exists and looks complete.'''
    if dest_path.exists() and dest_path.stat().st_size >= min_bytes:
        size_mb = dest_path.stat().st_size / 1e6
        print(f"  [SKIP]      {dest_path.name:20s} already exists ({size_mb:.1f} MB)")
        return "skipped"
    try:
        print(f"  [DOWNLOAD]  {dest_path.name:20s} <- {url}")
        tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
        urllib.request.urlretrieve(url, tmp_path)
        tmp_path.rename(dest_path)
        size_mb = dest_path.stat().st_size / 1e6
        print(f"  [OK]        {dest_path.name:20s} ({size_mb:.1f} MB)")
        return "downloaded"
    except (urllib.error.URLError, urllib.error.HTTPError) as e:
        if dest_path.exists():
            dest_path.unlink()
        raise RuntimeError(f"FAILED to download {url}: {e}") from e

download_if_missing(CHB12_BASE_URL + "chb12-summary.txt", chb12_summary_path)
summary_text = chb12_summary_path.read_text()

# NEW (CHB12-specific): CHB12's summary.txt contains one or more
# "Channels changed:" sections partway through the file list -- unlike
# CHB01/CHB24, which declare a single montage for the whole subject. We must
# track, for every File Name block, WHICH montage generation was in effect
# for it (the list of "Channel N: <label>" declarations most recently seen
# before that block), not just its seizure count.
initial_block = summary_text.split("File Name:")[0]
current_declared_channels = re.findall(r"Channel \d+:\s*(\S+)", initial_block)

tokens = re.split(r"(?=File Name: |Channels changed:)", summary_text)

file_declared_channels = {}   # fname -> list of declared channel labels (montage generation at that point)
file_seizure_counts = {}
file_seizure_intervals_raw = {}
montage_change_count = 0

for tok in tokens:
    if tok.startswith("Channels changed:"):
        current_declared_channels = re.findall(r"Channel \d+:\s*(\S+)", tok)
        montage_change_count += 1
        continue
    if not tok.startswith("File Name:"):
        continue
    name_match = re.search(r"File Name:\s*(\S+)", tok)
    fname = name_match.group(1)
    n_match = re.search(r"Number of Seizures in File:\s*(\d+)", tok)
    if not n_match:
        raise ValueError(f"{fname}: could not find seizure count in summary")
    n_seizures = int(n_match.group(1))
    starts = [int(s) for s in re.findall(r"Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s*seconds", tok)]
    ends = [int(s) for s in re.findall(r"Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s*seconds", tok)]
    if len(starts) != n_seizures or len(ends) != n_seizures:
        raise ValueError(f"{fname}: declared {n_seizures} seizures but parsed {len(starts)}/{len(ends)} times")

    file_declared_channels[fname] = list(current_declared_channels)
    file_seizure_counts[fname] = n_seizures
    file_seizure_intervals_raw[fname] = list(zip(starts, ends))

total_files = len(file_seizure_counts)
seizure_files_all = sorted(f for f, n in file_seizure_counts.items() if n > 0)
total_seizure_events_declared = sum(file_seizure_counts.values())

print(f"Files listed in chb12-summary.txt : {total_files}")
print(f"Montage generations found          : {montage_change_count + 1} "
      f"(1 initial + {montage_change_count} 'Channels changed' section(s))")
print(f"Files with >=1 declared seizure     : {len(seizure_files_all)}")
print(f"Total declared seizure events       : {total_seizure_events_declared}")
for fname in seizure_files_all:
    print(f"  {fname}: {file_seizure_counts[fname]} seizure(s)")


STEP 2: DOWNLOAD + PARSE chb12-summary.txt (SOURCE OF TRUTH)
  [SKIP]      chb12-summary.txt    already exists (0.0 MB)
Files listed in chb12-summary.txt : 24
Montage generations found          : 4 (1 initial + 3 'Channels changed' section(s))
Files with >=1 declared seizure     : 13
Total declared seizure events       : 40
  chb12_06.edf: 2 seizure(s)
  chb12_08.edf: 4 seizure(s)
  chb12_09.edf: 2 seizure(s)
  chb12_10.edf: 2 seizure(s)
  chb12_11.edf: 1 seizure(s)
  chb12_23.edf: 3 seizure(s)
  chb12_27.edf: 6 seizure(s)
  chb12_28.edf: 1 seizure(s)
  chb12_29.edf: 6 seizure(s)
  chb12_33.edf: 2 seizure(s)
  chb12_36.edf: 1 seizure(s)
  chb12_38.edf: 5 seizure(s)
  chb12_42.edf: 5 seizure(s)


### What did we just do, and why?

We parsed every `File Name` block in `chb12-summary.txt`, exactly as
`EEG_CHB24_preprocessing.ipynb` did for CHB24, but additionally tracked
which declared channel montage was in effect for each file by watching for
`"Channels changed:"` section headers. `chb12-summary.txt` (verified,
downloaded above) is the authoritative source for both seizure timing and
montage generation -- nothing here is assumed from prior knowledge of the
CHB-MIT dataset.

**Connection to the next step.** We now classify each seizure-containing
file's declared montage generation as compatible or incompatible with the
project's established 23-channel bipolar representation, and decide exactly
which EDFs to download.

In [4]:
print("\n" + "=" * 60)
print("STEP 3: CLASSIFY MONTAGE COMPATIBILITY (declared channels, before download)")
print("=" * 60)

# NEW (CHB12-specific). A montage generation is judged COMPATIBLE iff every
# one of the CANONICAL_EEG_CHANNELS names (the verified CHB01/CHB24 montage,
# duplicates included) is present among that generation's declared channels
# once "-" placeholder slots are removed. Extra channels beyond the
# canonical set (e.g. an added EOG channel) are tolerated -- they will be
# explicitly dropped before feature extraction (Step 8) -- but a MISSING
# canonical channel, or a fundamentally different referencing scheme (e.g.
# common-reference "-CS2" channels, or unreferenced single-electrode
# channels), makes the recording incompatible with our established feature
# representation and it is excluded, not silently reinterpreted.
#
# CANONICAL_EEG_CHANNELS holds MNE's already-deduplicated runtime names
# (e.g. "T8-P8-0"/"T8-P8-1" for the one duplicated label in this montage).
# For this DECLARED-text-based screen (raw summary.txt text, not yet loaded
# by MNE) we compare against the true declared name, so convert back:
canonical_declared_multiset = Counter(["T8-P8" if c.startswith("T8-P8-") else c for c in CANONICAL_EEG_CHANNELS])

def classify_montage(declared_channels):
    real = [c for c in declared_channels if c != "-"]
    real_multiset = Counter(real)
    missing = canonical_declared_multiset - real_multiset  # canonical channels absent here
    extra = real_multiset - canonical_declared_multiset    # channels present here beyond canonical
    compatible = len(missing) == 0
    return compatible, missing, extra

montage_reports = {}
for fname in seizure_files_all:
    compatible, missing, extra = classify_montage(file_declared_channels[fname])
    montage_reports[fname] = (compatible, missing, extra)

compatible_files = sorted(f for f, (ok, _, _) in montage_reports.items() if ok)
excluded_files = sorted(f for f, (ok, _, _) in montage_reports.items() if not ok)

print("Per seizure-containing file montage classification:")
for fname in seizure_files_all:
    compatible, missing, extra = montage_reports[fname]
    status = "COMPATIBLE" if compatible else "EXCLUDED"
    detail = f"extra={dict(extra)}" if compatible and extra else (f"missing={dict(missing)}" if not compatible else "")
    print(f"  {fname:14s} seizures={file_seizure_counts[fname]}  -> {status:10s} {detail}")

print(f"\n[OK] {len(compatible_files)} seizure-containing files are montage-compatible: {compatible_files}")
if excluded_files:
    print(f"\n[EXCLUDED] {len(excluded_files)} seizure-containing files are NOT montage-compatible "
          f"and will NOT be downloaded or processed:")
    for fname in excluded_files:
        _, missing, _ = montage_reports[fname]
        n_excluded_seizures = file_seizure_counts[fname]
        print(f"  {fname}: {n_excluded_seizures} seizure(s) excluded -- "
          f"missing canonical channels e.g. {list(missing)[:3]}{'...' if len(missing) > 3 else ''} "
          f"(different reference scheme, not a subset/superset of the verified montage)")

n_seizures_compatible = sum(file_seizure_counts[f] for f in compatible_files)
n_seizures_excluded = sum(file_seizure_counts[f] for f in excluded_files)
print(f"\nSeizure events in COMPATIBLE files: {n_seizures_compatible}")
print(f"Seizure events in EXCLUDED files  : {n_seizures_excluded}")

# Prioritize the most seizure-dense compatible recordings first (we are
# time-limited tonight); checkpointing (Step 8) makes this order, not just
# the final set, matter if the run is interrupted.
chb12_recording_ids = sorted(
    (Path(f).stem for f in compatible_files),
    key=lambda rid: -file_seizure_counts[f"{rid}.edf"],
)
print(f"\nDownload/processing order (most seizure-dense first): {chb12_recording_ids}")


STEP 3: CLASSIFY MONTAGE COMPATIBILITY (declared channels, before download)
Per seizure-containing file montage classification:
  chb12_06.edf   seizures=2  -> COMPATIBLE 
  chb12_08.edf   seizures=4  -> COMPATIBLE 
  chb12_09.edf   seizures=2  -> COMPATIBLE 
  chb12_10.edf   seizures=2  -> COMPATIBLE 
  chb12_11.edf   seizures=1  -> COMPATIBLE 
  chb12_23.edf   seizures=3  -> COMPATIBLE 
  chb12_27.edf   seizures=6  -> EXCLUDED   missing={'FP1-F7': 1, 'F7-T7': 1, 'T7-P7': 1, 'P7-O1': 1, 'FP1-F3': 1, 'F3-C3': 1, 'C3-P3': 1, 'P3-O1': 1, 'FP2-F4': 1, 'F4-C4': 1, 'C4-P4': 1, 'P4-O2': 1, 'FP2-F8': 1, 'F8-T8': 1, 'T8-P8': 2, 'P8-O2': 1, 'FZ-CZ': 1, 'CZ-PZ': 1, 'P7-T7': 1, 'T7-FT9': 1, 'FT9-FT10': 1, 'FT10-T8': 1}
  chb12_28.edf   seizures=1  -> EXCLUDED   missing={'FP1-F7': 1, 'F7-T7': 1, 'T7-P7': 1, 'P7-O1': 1, 'FP1-F3': 1, 'F3-C3': 1, 'C3-P3': 1, 'P3-O1': 1, 'FP2-F4': 1, 'F4-C4': 1, 'C4-P4': 1, 'P4-O2': 1, 'FP2-F8': 1, 'F8-T8': 1, 'T8-P8': 2, 'P8-O2': 1, 'FZ-CZ': 1, 'CZ-PZ': 1, 'P7-T7': 

### What did we just do, and why?

CHB12's summary file shows the montage changes twice partway through the
subject. We classified every seizure-containing file's DECLARED montage
(text only, no download yet) against `CANONICAL_EEG_CHANNELS`:

- **Compatible**: all 23 canonical bipolar channels are declared present
  (CHB12's original montage, and a later variant that adds one extra `LOC-
  ROC` channel -- tolerated, since it will be explicitly dropped before
  feature extraction).
- **Excluded**: a common-reference montage (`*-CS2` channels) and an
  unreferenced single-electrode montage (with an added `EKG1-CHIN` channel)
  -- neither is a bipolar-pair scheme compatible with our verified feature
  representation, so combining them would silently mix physically different
  quantities into the same "qEEG feature" columns.

This is why the notebook downloads only a SUBSET of CHB12's seizure-
containing EDFs, and reports exactly which ones and why -- as required.
Recordings to process are then ordered by declared seizure count,
descending, so the most valuable data is secured first under the
checkpointing scheme below.

**Connection to the next step.** Download only the compatible, prioritized
EDFs.

In [5]:
print("\n" + "=" * 60)
print("STEP 3B: RESTRICT TO ALREADY-DOWNLOADED SUBSET (time-limited stop)")
print("=" * 60)

# Time-limited decision (explicit user instruction): downloading was stopped
# after 5 of the 10 montage-compatible seizure EDFs finished, given slow
# network throughput. The remaining 5 compatible recordings are NOT required
# to exist -- this notebook proceeds on whatever is already on disk, and
# does not download anything further.
MIN_VALID_EDF_BYTES = 5_000_000

already_downloaded = []
not_fetched_in_time = []
for rec_id in chb12_recording_ids:
    p = chb_dir / f"{rec_id}.edf"
    if p.exists() and p.stat().st_size >= MIN_VALID_EDF_BYTES:
        already_downloaded.append(rec_id)
    else:
        not_fetched_in_time.append(rec_id)

print(f"Montage-compatible recordings identified in Step 3: {len(chb12_recording_ids)} {chb12_recording_ids}")
print(f"\n[USE]  Already downloaded, will be processed ({len(already_downloaded)}): {already_downloaded}")
for rec_id in already_downloaded:
    print(f"    {rec_id}: {file_seizure_counts[f'{rec_id}.edf']} seizure(s) declared")
print(f"\n[SKIP -- time-limited, NOT downloaded, NOT required] "
      f"({len(not_fetched_in_time)}): {not_fetched_in_time}")
for rec_id in not_fetched_in_time:
    print(f"    {rec_id}: {file_seizure_counts[f'{rec_id}.edf']} seizure(s) declared -- excluded "
          f"only because of tonight's time limit, NOT a montage/quality problem")

if not already_downloaded:
    raise RuntimeError("No CHB12 recordings are available on disk -- nothing to process.")

n_seizures_time_excluded = sum(file_seizure_counts[f"{r}.edf"] for r in not_fetched_in_time)
n_seizures_available = sum(file_seizure_counts[f"{r}.edf"] for r in already_downloaded)
print(f"\nSeizure events in the available subset      : {n_seizures_available}")
print(f"Seizure events left out (time-limited, not montage): {n_seizures_time_excluded}")

# From here on, the rest of this notebook operates ONLY on the available subset.
chb12_recording_ids = already_downloaded


STEP 3B: RESTRICT TO ALREADY-DOWNLOADED SUBSET (time-limited stop)
Montage-compatible recordings identified in Step 3: 10 ['chb12_38', 'chb12_42', 'chb12_08', 'chb12_23', 'chb12_06', 'chb12_09', 'chb12_10', 'chb12_33', 'chb12_11', 'chb12_36']

[USE]  Already downloaded, will be processed (5): ['chb12_38', 'chb12_42', 'chb12_08', 'chb12_23', 'chb12_06']
    chb12_38: 5 seizure(s) declared
    chb12_42: 5 seizure(s) declared
    chb12_08: 4 seizure(s) declared
    chb12_23: 3 seizure(s) declared
    chb12_06: 2 seizure(s) declared

[SKIP -- time-limited, NOT downloaded, NOT required] (5): ['chb12_09', 'chb12_10', 'chb12_33', 'chb12_11', 'chb12_36']
    chb12_09: 2 seizure(s) declared -- excluded only because of tonight's time limit, NOT a montage/quality problem
    chb12_10: 2 seizure(s) declared -- excluded only because of tonight's time limit, NOT a montage/quality problem
    chb12_33: 2 seizure(s) declared -- excluded only because of tonight's time limit, NOT a montage/quality prob

### What did we just do, and why?

Tonight's run was explicitly stopped after 5 of the 10 montage-compatible
CHB12 EDFs finished downloading (slow network, time-limited session) --one
file that was mid-download when the stop was issued was terminated and its
incomplete partial file deleted; the other 4 already-completed files were
left untouched. This is a **time constraint**, not a montage or data-quality
exclusion (that decision was already made, separately, in Step 3): the 5
recordings left out here are still montage-compatible and could be added in
a future run.

`chb12_recording_ids` is reassigned to exactly the recordings already on
disk, so every step downstream (seizure-interval parsing, per-EDF
processing, validation, and the execution report) operates only on this
subset and does not require or attempt to fetch the rest.

**Connection to the next step.** Download step becomes a pure verification
(no network calls) against this restricted, already-on-disk set.

In [6]:
print("\n" + "=" * 60)
print("STEP 4: VERIFY THE ALREADY-DOWNLOADED CHB12 EDFs ARE PRESENT (no network calls)")
print("=" * 60)

# Time-limited run: downloading is deliberately NOT performed here. Every
# recording in chb12_recording_ids was already confirmed present on disk in
# Step 3B; this step only re-verifies that and reports sizes -- if anything
# is unexpectedly missing, this raises rather than silently downloading it.

chb12_files = {rec_id: chb_dir / f"{rec_id}.edf" for rec_id in chb12_recording_ids}

total_bytes = 0
missing = []
for rec_id, dest_path in chb12_files.items():
    if not dest_path.exists():
        missing.append(rec_id)
        continue
    size_mb = dest_path.stat().st_size / 1e6
    total_bytes += dest_path.stat().st_size
    print(f"  [OK] {dest_path.name:20s} present ({size_mb:.1f} MB)")

if missing:
    raise RuntimeError(
        f"Missing EDFs that Step 3B expected to already be on disk: {missing}. "
        f"This run does not download -- restore the files or rerun Step 3B's filter."
    )

print(f"\nRequested (already-downloaded) files: {len(chb12_files)}")
print(f"Total size on disk: {total_bytes / 1e6:.1f} MB")
print("\n[OK] All recordings in the time-limited subset are present on disk (no downloads performed)")


STEP 4: VERIFY THE ALREADY-DOWNLOADED CHB12 EDFs ARE PRESENT (no network calls)
  [OK] chb12_38.edf         present (53.5 MB)
  [OK] chb12_42.edf         present (53.5 MB)
  [OK] chb12_08.edf         present (51.6 MB)
  [OK] chb12_23.edf         present (51.6 MB)
  [OK] chb12_06.edf         present (51.7 MB)

Requested (already-downloaded) files: 5
Total size on disk: 261.9 MB

[OK] All recordings in the time-limited subset are present on disk (no downloads performed)


### What did we just do, and why?

Idempotent, restartable download identical in mechanics to CHB24's: each
file is checked individually before fetching, so a rerun after a crash
simply skips whatever already downloaded successfully. Only the montage-
compatible seizure-containing recordings identified above are fetched --
still no full-subject download.

**Connection to the next step.** Before processing, we verify each
downloaded file's ACTUAL runtime channel structure (not just its declared
text) against `CANONICAL_EEG_CHANNELS`, and determine exactly which
channels must be dropped from each file so that only the verified 23-
channel EEG set reaches the (verbatim, unmodified) preprocessing functions.

In [7]:
print("\n" + "=" * 60)
print("STEP 5: VERIFY EDF METADATA AND MONTAGE AT RUNTIME (BEFORE PROCESSING)")
print("=" * 60)

# Lightweight (preload=False) header-only inspection, exactly like CHB24
# Step 4, but additionally computing the exact per-file channel-drop list
# needed to reconcile CHB12's montage with CANONICAL_EEG_CHANNELS.

canonical_set = set(CANONICAL_EEG_CHANNELS)
canonical_multiset = Counter(CANONICAL_EEG_CHANNELS)

edf_metadata = {}
channels_to_drop_by_rec = {}

for rec_id, path in chb12_files.items():
    raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
    sfreq = float(raw_header.info["sfreq"])
    ch_names = list(raw_header.ch_names)
    duration_sec = float(raw_header.n_times) / sfreq

    # NEW (CHB12-specific): drop every channel whose (possibly MNE-
    # deduplicated) name is not one of CANONICAL_EEG_CHANNELS. This is
    # name-based, not position-based, so it is robust to CHB12's different
    # channel ORDER and to whatever exact suffix MNE assigns to duplicate
    # placeholder ("-") channels.
    drop_names = [ch for ch in ch_names if ch not in canonical_set]
    keep_names = [ch for ch in ch_names if ch in canonical_set]

    edf_metadata[rec_id] = {
        "sfreq": sfreq, "ch_names": ch_names, "duration_sec": duration_sec,
        "drop_names": drop_names, "keep_names": keep_names,
    }
    channels_to_drop_by_rec[rec_id] = drop_names

    keep_multiset = Counter(keep_names)
    montage_ok = (keep_multiset == canonical_multiset)

    print(f"  {rec_id:12s}: sfreq={sfreq:6.1f}Hz, total_ch={len(ch_names):2d}, "
          f"dropped={len(drop_names):2d} {drop_names}, kept={len(keep_names):2d}, "
          f"duration={duration_sec:7.1f}s, montage_matches_canonical={montage_ok}")

    if not np.isclose(sfreq, expected_chb_sfreq):
        raise ValueError(f"{rec_id}: sfreq {sfreq}Hz does not match verified {expected_chb_sfreq}Hz")
    if not montage_ok:
        raise ValueError(
            f"{rec_id}: after dropping non-canonical channels, remaining channel "
            f"multiset does not exactly match CANONICAL_EEG_CHANNELS. "
            f"kept={sorted(keep_multiset.elements())} expected={sorted(canonical_multiset.elements())}"
        )
    del raw_header

print(f"\n[OK] All {len(chb12_files)} CHB12 recordings verified: uniform {expected_chb_sfreq}Hz sampling, "
      f"and (after dropping non-canonical channels) an exact match to the {len(CANONICAL_EEG_CHANNELS)}-channel "
      f"CANONICAL_EEG_CHANNELS multiset used throughout this project.")


STEP 5: VERIFY EDF METADATA AND MONTAGE AT RUNTIME (BEFORE PROCESSING)


  chb12_38    : sfreq= 256.0Hz, total_ch=29, dropped= 6 ['--0', '--1', '--2', '--3', '--4', 'LOC-ROC'], kept=23, duration= 3600.0s, montage_matches_canonical=True
  chb12_42    : sfreq= 256.0Hz, total_ch=29, dropped= 6 ['--0', '--1', '--2', '--3', '--4', 'LOC-ROC'], kept=23, duration= 3600.0s, montage_matches_canonical=True
  chb12_08    : sfreq= 256.0Hz, total_ch=28, dropped= 5 ['--0', '--1', '--2', '--3', '--4'], kept=23, duration= 3600.0s, montage_matches_canonical=True
  chb12_23    : sfreq= 256.0Hz, total_ch=28, dropped= 5 ['--0', '--1', '--2', '--3', '--4'], kept=23, duration= 3600.0s, montage_matches_canonical=True


  chb12_06    : sfreq= 256.0Hz, total_ch=28, dropped= 5 ['--0', '--1', '--2', '--3', '--4'], kept=23, duration= 3606.0s, montage_matches_canonical=True

[OK] All 5 CHB12 recordings verified: uniform 256.0Hz sampling, and (after dropping non-canonical channels) an exact match to the 23-channel CANONICAL_EEG_CHANNELS multiset used throughout this project.


/tmp/ipykernel_53220/3462734830.py:16: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '-'}. Applying running numbers for duplicates.
  raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
/tmp/ipykernel_53220/3462734830.py:16: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
/tmp/ipykernel_53220/3462734830.py:16: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '-'}. Applying running numbers for duplicates.
  raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
/tmp/ipykernel_53220/3462734830.py:16: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
/tmp/ipykernel_53220/3462734830.py:16: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '-'}. A

### What did we just do, and why?

For every downloaded CHB12 file we opened its real EDF header (no signal
data loaded yet) and computed exactly which channels are NOT part of the
verified 23-channel montage (`drop_names`) -- these are the placeholder `-`
slots (however MNE ends up naming their duplicates) and, for the later
CHB12 recordings, the extra `LOC-ROC` channel. After removing them, we
assert the REMAINING channels are an exact multiset match (same 23 names,
duplicate `T8-P8-0`/`T8-P8-1` included) to `CANONICAL_EEG_CHANNELS` -- not
just "close enough". Any recording that failed this check would stop the
notebook here rather than silently producing features that mean something
different.

**Why name-based dropping, not re-running the montage classification from
Step 3.** Step 3 worked from the summary file's declared text (fast, no
download needed, used to decide what to fetch). This step re-verifies
against the actual bytes of the downloaded EDF, which is the real
correctness guarantee -- consistent with this project's evidence hierarchy
(runtime evidence over documentation).

**Connection to the next step.** With per-recording drop lists computed and
verified, we load the same preprocessing helper functions used for CHB01
and CHB24, completely unmodified.

In [8]:
print("\n" + "=" * 60)
print("STEP 6: VERIFIED PREPROCESSING HELPERS (copied verbatim, not reimplemented)")
print("=" * 60)

# Everything in this cell is copied VERBATIM from EEG_CHB24_preprocessing.ipynb
# (itself verbatim from EEG_preprocessing.ipynb). Do not modify this cell
# independently of the source notebooks.

def preprocess_eeg(raw, dataset_name, l_freq=FILTER_LOW_HZ, h_freq=FILTER_HIGH_HZ):
    '''
    Select EEG-typed channels, apply band-pass filter, validate.

    Returns:
        eeg_raw: MNE Raw object (EEG channels only, filtered)
        data: NumPy array (channels, samples)
        sfreq: Sampling frequency
    '''
    eeg_raw = raw.copy().pick(picks="eeg")

    if len(eeg_raw.ch_names) == 0:
        raise ValueError(f"{dataset_name}: No EEG channels found")

    sfreq = float(eeg_raw.info["sfreq"])
    nyquist = sfreq / 2.0

    if h_freq >= nyquist:
        raise ValueError(
            f"{dataset_name}: filter cutoff {h_freq}Hz >= Nyquist {nyquist}Hz "
            f"(sfreq={sfreq}Hz)"
        )

    eeg_raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)

    data = eeg_raw.get_data()

    if data.ndim != 2:
        raise ValueError(f"{dataset_name}: Expected 2D data, got {data.shape}")

    if not np.isfinite(data).all():
        raise ValueError(f"{dataset_name}: Data contains NaN/Inf after filtering")

    print(f"{dataset_name:16s}: channels={len(eeg_raw.ch_names):2d}, "
          f"sfreq={sfreq:6.1f}Hz, shape={data.shape}, "
          f"filter=[{l_freq:.1f}, {h_freq:.1f}]Hz")

    return eeg_raw, data, sfreq


def make_epochs(data, sfreq, win_sec=EPOCH_DURATION_SEC, overlap=EPOCH_OVERLAP):
    '''Create fixed-duration, non-overlapping epochs from continuous data.'''
    if data.ndim != 2:
        raise ValueError(f"Expected 2D data, got {data.shape}")

    if not (0.0 <= overlap < 1.0):
        raise ValueError("overlap must be in [0, 1)")

    win_samp = int(round(win_sec * sfreq))
    step = int(round(win_samp * (1 - overlap)))

    if win_samp <= 0:
        raise ValueError("win_sec and sfreq must produce positive window size")

    if step <= 0:
        raise ValueError("overlap is too large")

    n_samples = data.shape[1]

    if n_samples < win_samp:
        raise ValueError(
            f"Recording has {n_samples} samples, but epoch requires {win_samp}"
        )

    starts = np.arange(0, n_samples - win_samp + 1, step)

    epochs = np.stack(
        [data[:, start:start + win_samp] for start in starts],
        axis=0
    )

    return epochs


def validate_epochs(name, epochs, sfreq, win_sec=EPOCH_DURATION_SEC,
                   expected_channels=None):
    '''Validate epoch structure and content (shape, channels, finiteness).'''
    expected_samples = int(round(win_sec * sfreq))

    if epochs.ndim != 3:
        raise ValueError(f"{name}: Expected 3D, got {epochs.shape}")

    if epochs.shape[2] != expected_samples:
        raise ValueError(
            f"{name}: Expected {expected_samples} samples/epoch, got {epochs.shape[2]}"
        )

    if expected_channels is not None:
        if epochs.shape[1] != expected_channels:
            raise ValueError(
                f"{name}: Expected {expected_channels} channels, got {epochs.shape[1]}"
            )

    if not np.isfinite(epochs).all():
        raise ValueError(f"{name}: Contains NaN/Inf values")

    print(f"{name:16s}: epochs={epochs.shape[0]:5d}, channels={epochs.shape[1]:2d}, "
          f"samples={epochs.shape[2]:5d}, duration={win_sec:.0f}s")


def bandpower(x, sfreq, fmin, fmax, nperseg=None):
    '''Absolute band power in [fmin, fmax] via Welch PSD (trapezoidal integration).'''
    if nperseg is None:
        nperseg = min(len(x), int(2 * sfreq))
    freqs, psd = welch(x, fs=sfreq, nperseg=nperseg)
    mask = (freqs >= fmin) & (freqs <= fmax)
    if not np.any(mask):
        return np.nan
    return np.trapezoid(psd[mask], freqs[mask])


def spectral_entropy_welch(x, sfreq, fmin=1.0, fmax=40.0, nperseg=None, eps=1e-12):
    '''Shannon entropy of the normalized Welch PSD, normalized to [0, 1].'''
    if nperseg is None:
        nperseg = min(len(x), int(2 * sfreq))
    freqs, psd = welch(x, fs=sfreq, nperseg=nperseg)
    mask = (freqs >= fmin) & (freqs <= fmax)
    psd_band = np.maximum(psd[mask], eps)
    p = psd_band / psd_band.sum()
    ent = -np.sum(p * np.log(p))
    return ent / np.log(len(p))


def extract_features(epochs, sfreq):
    '''Extract qEEG spectral features from epochs, averaged across channels.'''
    features = []
    eps = 1e-12

    for ep in epochs:
        std = np.std(ep)
        var = np.var(ep)
        rms = np.sqrt(np.mean(ep**2))

        delta = np.mean([bandpower(ch, sfreq, 0.5, 4) for ch in ep])
        theta = np.mean([bandpower(ch, sfreq, 4, 8) for ch in ep])
        alpha = np.mean([bandpower(ch, sfreq, 8, 13) for ch in ep])
        beta = np.mean([bandpower(ch, sfreq, 13, 30) for ch in ep])

        total_power = delta + theta + alpha + beta + eps
        rel_delta = delta / total_power
        rel_theta = theta / total_power
        rel_alpha = alpha / total_power
        rel_beta = beta / total_power

        theta_alpha_ratio = theta / (alpha + eps)
        delta_beta_ratio = delta / (beta + eps)

        ent = np.mean([spectral_entropy_welch(ch, sfreq) for ch in ep])

        features.append({
            "std": std, "var": var, "rms": rms,
            "delta": delta, "theta": theta, "alpha": alpha, "beta": beta,
            "rel_delta": rel_delta, "rel_theta": rel_theta,
            "rel_alpha": rel_alpha, "rel_beta": rel_beta,
            "theta_alpha_ratio": theta_alpha_ratio,
            "delta_beta_ratio": delta_beta_ratio,
            "entropy": ent,
        })

    return pd.DataFrame(features)


def parse_chb_summary(summary_path, known_edf_filenames):
    '''
    Parse a CHB-MIT *-summary.txt file into {edf_filename: [(start_s, end_s), ...]}.
    Only entries whose File Name is in `known_edf_filenames` are kept.
    '''
    text = summary_path.read_text()
    blocks = re.split(r"(?=File Name: )", text)

    annotations = {}
    for block in blocks:
        name_match = re.search(r"File Name:\s*(\S+)", block)
        if not name_match:
            continue
        fname = name_match.group(1)
        if fname not in known_edf_filenames:
            continue

        n_match = re.search(r"Number of Seizures in File:\s*(\d+)", block)
        if not n_match:
            raise ValueError(f"{fname}: could not find seizure count in summary")
        n_seizures = int(n_match.group(1))

        starts = [int(s) for s in re.findall(r"Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s*seconds", block)]
        ends = [int(s) for s in re.findall(r"Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s*seconds", block)]

        if len(starts) != n_seizures or len(ends) != n_seizures:
            raise ValueError(
                f"{fname}: declared {n_seizures} seizures but parsed "
                f"{len(starts)} start / {len(ends)} end times"
            )

        annotations[fname] = list(zip(starts, ends))

    missing = set(known_edf_filenames) - set(annotations)
    if missing:
        raise ValueError(f"No summary entry found for: {missing}")

    return annotations


def label_seizure_epochs(t_start_sec, t_end_sec, intervals):
    '''
    y_seizure = 1 iff [t_start_sec, t_end_sec) intersects any annotated
    seizure interval, using true interval intersection.
    '''
    y = np.zeros(len(t_start_sec), dtype=int)
    for i, (t_start, t_end) in enumerate(zip(t_start_sec, t_end_sec)):
        for seiz_start, seiz_end in intervals:
            if t_start < seiz_end and t_end > seiz_start:
                y[i] = 1
                break
    return y


def add_temporal_metadata(df, win_sec=EPOCH_DURATION_SEC):
    '''Add per-recording epoch index and elapsed-time columns.'''
    df = df.copy()

    if "recording_id" not in df.columns:
        raise ValueError("Expected 'recording_id' column")

    df["epoch_idx"] = df.groupby("recording_id").cumcount()
    df["t_start_sec"] = df["epoch_idx"] * win_sec
    df["t_center_sec"] = df["t_start_sec"] + win_sec / 2.0
    df["t_end_sec"] = df["t_start_sec"] + win_sec

    return df

print("[OK] Verified helper functions loaded (verbatim copies):")
for fn in [preprocess_eeg, make_epochs, validate_epochs, bandpower,
           spectral_entropy_welch, extract_features, parse_chb_summary,
           label_seizure_epochs, add_temporal_metadata]:
    print(f"  - {fn.__name__}")


STEP 6: VERIFIED PREPROCESSING HELPERS (copied verbatim, not reimplemented)
[OK] Verified helper functions loaded (verbatim copies):
  - preprocess_eeg
  - make_epochs
  - validate_epochs
  - bandpower
  - spectral_entropy_welch
  - extract_features
  - parse_chb_summary
  - label_seizure_epochs
  - add_temporal_metadata


In [9]:
print("\n" + "=" * 60)
print("STEP 7: PARSE CHB12 SEIZURE INTERVALS (restricted to compatible recordings)")
print("=" * 60)

chb12_edf_filenames = {rec_id: chb12_files[rec_id].name for rec_id in chb12_recording_ids}

chb12_seizure_intervals_by_file = parse_chb_summary(
    chb12_summary_path,
    known_edf_filenames=set(chb12_edf_filenames.values()),
)
chb12_seizure_intervals = {
    rec_id: chb12_seizure_intervals_by_file[fname]
    for rec_id, fname in chb12_edf_filenames.items()
}

n_seizure_events_total = sum(len(v) for v in chb12_seizure_intervals.values())
print(f"\nParsed from {chb12_summary_path.name} (compatible recordings only):")
for rec_id, intervals in chb12_seizure_intervals.items():
    for start, end in intervals:
        print(f"  {rec_id}: seizure {start}-{end}s (duration {end-start}s)")

print(f"\n[OK] Total seizure events parsed (compatible recordings): {n_seizure_events_total}")
if n_seizure_events_total != n_seizures_available:
    raise ValueError(
        f"Parsed seizure count ({n_seizure_events_total}) != Step 3B's available-subset "
        f"declared count ({n_seizures_available})"
    )


STEP 7: PARSE CHB12 SEIZURE INTERVALS (restricted to compatible recordings)

Parsed from chb12-summary.txt (compatible recordings only):
  chb12_38: seizure 1548-1573s (duration 25s)
  chb12_38: seizure 2798-2821s (duration 23s)
  chb12_38: seizure 2966-3009s (duration 43s)
  chb12_38: seizure 3146-3201s (duration 55s)
  chb12_38: seizure 3364-3410s (duration 46s)
  chb12_42: seizure 699-750s (duration 51s)
  chb12_42: seizure 945-973s (duration 28s)
  chb12_42: seizure 1170-1199s (duration 29s)
  chb12_42: seizure 1676-1701s (duration 25s)
  chb12_42: seizure 2213-2236s (duration 23s)
  chb12_08: seizure 1426-1439s (duration 13s)
  chb12_08: seizure 1591-1614s (duration 23s)
  chb12_08: seizure 1957-1977s (duration 20s)
  chb12_08: seizure 2798-2824s (duration 26s)
  chb12_23: seizure 253-333s (duration 80s)
  chb12_23: seizure 425-522s (duration 97s)
  chb12_23: seizure 630-670s (duration 40s)
  chb12_06: seizure 1665-1726s (duration 61s)
  chb12_06: seizure 3415-3447s (duration 32s

### What did we just do, and why?

Identical mechanics to CHB24 Step 6: `parse_chb_summary` is reused
unmodified, restricted to the montage-compatible recordings only, and
cross-checked against the count already established in Step 3.

**Connection to the next step.** Process each compatible EDF one at a time
-- download already done, montage already verified and drop-lists already
computed -- with a checkpoint written after every recording.

In [10]:
print("\n" + "=" * 60)
print("STEP 8: INCREMENTAL PER-EDF PROCESSING (memory-conservative, checkpointed)")
print("=" * 60)

# One EDF at a time: load -> drop non-canonical channels (NEW, CHB12-only
# step) -> preprocess -> epoch -> extract 14 qEEG features -> checkpoint ->
# release RAM -> gc.collect() -> next EDF. A checkpoint parquet is written
# immediately after each recording and, on rerun, is loaded and schema-
# validated instead of reprocessing that EDF.

checkpoint_dir = output_dir / "preprocessed_data" / "chb12_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

SUBJECT_ID = 12
DATASET_NAME = "chb"
EXPECTED_CHECKPOINT_COLS = set(qeeg_feature_cols) | {"recording_id", "dataset", "subject_id"}

def checkpoint_is_valid(path):
    try:
        chk = pd.read_parquet(path)
    except Exception as e:
        print(f"    [CHECKPOINT INVALID] {path.name}: unreadable ({e})")
        return False, None
    if len(chk) == 0:
        print(f"    [CHECKPOINT INVALID] {path.name}: 0 rows")
        return False, None
    if not EXPECTED_CHECKPOINT_COLS.issubset(chk.columns):
        print(f"    [CHECKPOINT INVALID] {path.name}: missing columns "
              f"{EXPECTED_CHECKPOINT_COLS - set(chk.columns)}")
        return False, None
    if not np.isfinite(chk[qeeg_feature_cols].to_numpy()).all():
        print(f"    [CHECKPOINT INVALID] {path.name}: NaN/Inf in qEEG features")
        return False, None
    return True, chk

feat_chb12_parts = []

for rec_id in chb12_recording_ids:
    checkpoint_path = checkpoint_dir / f"features_{rec_id}.parquet"

    if checkpoint_path.exists():
        valid, chk = checkpoint_is_valid(checkpoint_path)
        if valid:
            print(f"\n[CHECKPOINT SKIP] {rec_id}: {checkpoint_path.name} exists and validates "
                  f"({len(chk)} epochs)")
            feat_chb12_parts.append(chk)
            continue
        else:
            print(f"\n[CHECKPOINT REJECTED] {rec_id}: reprocessing from source EDF")

    print(f"\nProcessing {rec_id} ...")
    path = chb12_files[rec_id]

    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)

    # NEW (CHB12-specific, minimal, verified in Step 5): drop channels not
    # in the canonical 23-channel montage BEFORE calling the verbatim
    # preprocess_eeg() -- this is the only deviation from the CHB01/CHB24
    # pipeline, and it only decides which raw channels are handed to that
    # unmodified function.
    drop_names = channels_to_drop_by_rec[rec_id]
    if drop_names:
        raw.drop_channels(drop_names)

    eeg_raw, data, sfreq = preprocess_eeg(raw, rec_id)

    if Counter(eeg_raw.ch_names) != canonical_multiset:
        raise ValueError(
            f"{rec_id}: post-drop, post-pick EEG channel set does not match "
            f"CANONICAL_EEG_CHANNELS -- got {sorted(eeg_raw.ch_names)}"
        )

    epochs = make_epochs(data, sfreq)
    validate_epochs(rec_id, epochs, sfreq, expected_channels=len(CANONICAL_EEG_CHANNELS))

    feat = extract_features(epochs, sfreq)
    feat["recording_id"] = rec_id
    feat["dataset"] = DATASET_NAME
    feat["subject_id"] = SUBJECT_ID

    feat.to_parquet(checkpoint_path, index=False)
    feat_chb12_parts.append(feat)
    print(f"  [OK] {rec_id}: {feat.shape[0]} epochs -> checkpoint saved")

    del raw, eeg_raw, data, epochs
    gc.collect()

feat_chb12_all = pd.concat(feat_chb12_parts, ignore_index=True)

# recording_id convention: match existing "chb_s24_r01" pattern exactly.
feat_chb12_all["recording_id"] = feat_chb12_all["recording_id"].apply(
    lambda rid: f"chb_s{SUBJECT_ID:02d}_r{rid.split('_')[1]}"
)

print(f"\n[OK] All {len(chb12_recording_ids)} CHB12 recordings processed")
print(f"  Total epochs: {feat_chb12_all.shape[0]}")
print(f"  recording_id values: {sorted(feat_chb12_all['recording_id'].unique())}")


STEP 8: INCREMENTAL PER-EDF PROCESSING (memory-conservative, checkpointed)

Processing chb12_38 ...


/tmp/ipykernel_53220/834836040.py:54: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '-'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_53220/834836040.py:54: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


chb12_38        : channels=23, sfreq= 256.0Hz, shape=(23, 921600), filter=[0.5, 40.0]Hz


chb12_38        : epochs=  120, channels=23, samples= 7680, duration=30s


  [OK] chb12_38: 120 epochs -> checkpoint saved



Processing chb12_42 ...


... (12 outputs left out)

/tmp/ipykernel_53220/834836040.py:54: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '-'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_53220/834836040.py:54: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


chb12_06        : channels=23, sfreq= 256.0Hz, shape=(23, 923136), filter=[0.5, 40.0]Hz


chb12_06        : epochs=  120, channels=23, samples= 7680, duration=30s


  [OK] chb12_06: 120 epochs -> checkpoint saved

[OK] All 5 CHB12 recordings processed
  Total epochs: 600
  recording_id values: ['chb_s12_r06', 'chb_s12_r08', 'chb_s12_r23', 'chb_s12_r38', 'chb_s12_r42']


### What did we just do, and why?

Each compatible CHB12 EDF was loaded exactly once, had its non-canonical
channels dropped (the one CHB12-specific adaptation, verified in Step 5),
and was then handed to the completely unmodified `preprocess_eeg` /
`make_epochs` / `validate_epochs` / `extract_features` pipeline. A parquet
checkpoint was written immediately after each recording, and the loop
releases the raw/epoch arrays and calls `gc.collect()` before continuing --
so a crash never loses more than the recording currently in progress, and a
rerun skips every recording whose checkpoint already exists and validates
(right down to checking for NaN/Inf and the expected columns, not just file
existence).

**Connection to the next step.** With per-epoch features in place, add the
same temporal metadata, `y_seizure` labels, and forecasting-horizon columns
used for CHB01/CHB24.

In [11]:
print("\n" + "=" * 60)
print("STEP 9: TEMPORAL METADATA + SEIZURE LABELING + FORECASTING HORIZON")
print("=" * 60)

feat_chb12_all = add_temporal_metadata(feat_chb12_all)

chb12_seizure_intervals_final = {
    f"chb_s{SUBJECT_ID:02d}_r{rec_id.split('_')[1]}": intervals
    for rec_id, intervals in chb12_seizure_intervals.items()
}

feat_chb12_all["y_seizure"] = 0
for recording_id, intervals in chb12_seizure_intervals_final.items():
    mask = feat_chb12_all["recording_id"] == recording_id
    feat_chb12_all.loc[mask, "y_seizure"] = label_seizure_epochs(
        feat_chb12_all.loc[mask, "t_start_sec"],
        feat_chb12_all.loc[mask, "t_end_sec"],
        intervals,
    )

n_seizure_positive = int(feat_chb12_all["y_seizure"].sum())
n_seizure_negative = len(feat_chb12_all) - n_seizure_positive
assert set(feat_chb12_all["y_seizure"].unique()) <= {0, 1}, "y_seizure must be binary"

print(f"y_seizure labeled: {n_seizure_positive} positive / {n_seizure_negative} negative epochs")

def compute_time_to_next_seizure(df, seizure_intervals_by_recording):
    df = df.copy()
    df["next_seizure_start_sec"] = np.nan
    df["time_to_next_seizure_sec"] = np.nan

    for recording_id, intervals in seizure_intervals_by_recording.items():
        mask = df["recording_id"] == recording_id
        if not mask.any():
            continue
        onsets = np.sort(np.array([s for s, _ in intervals], dtype=float))
        t_end = df.loc[mask, "t_end_sec"].to_numpy()

        idx = np.searchsorted(onsets, t_end, side="left")
        next_onset = np.where(idx < len(onsets), onsets[np.clip(idx, 0, len(onsets) - 1)], np.nan)

        df.loc[mask, "next_seizure_start_sec"] = next_onset
        df.loc[mask, "time_to_next_seizure_sec"] = next_onset - t_end

    return df

feat_chb12_all = compute_time_to_next_seizure(feat_chb12_all, chb12_seizure_intervals_final)

n_with_future_seizure = int(feat_chb12_all["next_seizure_start_sec"].notna().sum())
n_no_future_seizure = int(feat_chb12_all["next_seizure_start_sec"].isna().sum())
print(f"Epochs with a future seizure in-recording: {n_with_future_seizure}")
print(f"Epochs with NO future seizure (NaN)      : {n_no_future_seizure}")

finite_ttn = feat_chb12_all["time_to_next_seizure_sec"].dropna()
assert (finite_ttn >= 0).all(), "time_to_next_seizure_sec must be non-negative"
assert feat_chb12_all["time_to_next_seizure_sec"].isna().sum() == n_no_future_seizure

print("\n[OK] Temporal metadata, y_seizure, and forecasting-horizon columns added")


STEP 9: TEMPORAL METADATA + SEIZURE LABELING + FORECASTING HORIZON


y_seizure labeled: 41 positive / 559 negative epochs
Epochs with a future seizure in-recording: 412
Epochs with NO future seizure (NaN)      : 188

[OK] Temporal metadata, y_seizure, and forecasting-horizon columns added


In [12]:
print("\n" + "=" * 60)
print("STEP 10: SAVE CHB12 PROCESSED FEATURE TABLE")
print("=" * 60)

output_csv_path = output_dir / "features_chb12_forecasting.csv"
output_parquet_path = output_dir / "features_chb12_forecasting.parquet"

feat_chb12_all.to_csv(output_csv_path, index=False)
feat_chb12_all.to_parquet(output_parquet_path, index=False)

print(f"Saved: {output_parquet_path}")
print(f"Saved: {output_csv_path}")
print(f"Shape: {feat_chb12_all.shape}")
print(f"Columns: {list(feat_chb12_all.columns)}")


STEP 10: SAVE CHB12 PROCESSED FEATURE TABLE
Saved: /home/evgenija/projects/EEG_timeseries_project/outputs/features_chb12_forecasting.parquet
Saved: /home/evgenija/projects/EEG_timeseries_project/outputs/features_chb12_forecasting.csv
Shape: (600, 24)
Columns: ['std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy', 'recording_id', 'dataset', 'subject_id', 'epoch_idx', 't_start_sec', 't_center_sec', 't_end_sec', 'y_seizure', 'next_seizure_start_sec', 'time_to_next_seizure_sec']


### What did we just do, and why?

Same temporal-metadata / labeling / forecasting-horizon logic as CHB24
(`add_temporal_metadata`, `label_seizure_epochs`, and the new-but-
already-verified `compute_time_to_next_seizure` helper, all unchanged), and
the resulting table saved to `outputs/features_chb12_forecasting.parquet`
(+ CSV), matching the CHB24 output naming convention exactly.

**Connection to the next step.** Validate this table against the existing
CHB24 table before anything downstream is allowed to use it.

In [13]:
print("\n" + "=" * 60)
print("STEP 11: COMPATIBILITY VALIDATION vs EXISTING CHB24 TABLE")
print("=" * 60)

existing_cols = set(existing_chb24_df.columns)
new_cols = set(feat_chb12_all.columns)

feature_cols_identical = set(qeeg_feature_cols).issubset(existing_cols) and \
                          set(qeeg_feature_cols).issubset(new_cols)

print(f"Existing CHB24 columns ({len(existing_cols)}): {sorted(existing_cols)}")
print(f"\nNew CHB12 columns ({len(new_cols)}): {sorted(new_cols)}")
print(f"\nqEEG feature columns identical? {feature_cols_identical}")

only_in_existing = sorted(existing_cols - new_cols)
only_in_new = sorted(new_cols - existing_cols)
print(f"\nColumns only in existing CHB24 table: {only_in_existing}")
print(f"Columns only in new CHB12 table     : {only_in_new}")

print("\ndtype comparison for shared columns:")
dtype_mismatches = []
for col in sorted(existing_cols & new_cols):
    dt_old = existing_chb24_df[col].dtype
    dt_new = feat_chb12_all[col].dtype
    compatible = pd.api.types.is_numeric_dtype(dt_old) == pd.api.types.is_numeric_dtype(dt_new)
    if not compatible:
        dtype_mismatches.append((col, dt_old, dt_new))
    print(f"  {col:24s} existing={str(dt_old):10s} new={str(dt_new):10s} compatible={compatible}")

feature_table_no_forecasting = feat_chb12_all.drop(
    columns=["next_seizure_start_sec", "time_to_next_seizure_sec"]
)
n_nan_new = int(feature_table_no_forecasting.isna().sum().sum())
n_inf_new = int(np.isinf(
    feature_table_no_forecasting.select_dtypes(include=[np.number])
).sum().sum())
print(f"\nMissing values in new table (excluding intentional NaN forecasting cols): {n_nan_new}")
print(f"Infinite values in new table: {n_inf_new}")

if not feature_cols_identical:
    raise ValueError("CHB12 qEEG feature columns are not identical to CHB24's")
if only_in_existing or only_in_new:
    raise ValueError(f"Schema mismatch vs CHB24: only_in_existing={only_in_existing}, only_in_new={only_in_new}")
if dtype_mismatches:
    raise ValueError(f"dtype incompatibilities found: {dtype_mismatches}")
if n_inf_new != 0:
    raise ValueError(f"CHB12 feature table contains {n_inf_new} infinite values")

print("\n[OK] CHB12 table is schema- and dtype-IDENTICAL to the existing CHB24 table")


STEP 11: COMPATIBILITY VALIDATION vs EXISTING CHB24 TABLE
Existing CHB24 columns (24): ['alpha', 'beta', 'dataset', 'delta', 'delta_beta_ratio', 'entropy', 'epoch_idx', 'next_seizure_start_sec', 'recording_id', 'rel_alpha', 'rel_beta', 'rel_delta', 'rel_theta', 'rms', 'std', 'subject_id', 't_center_sec', 't_end_sec', 't_start_sec', 'theta', 'theta_alpha_ratio', 'time_to_next_seizure_sec', 'var', 'y_seizure']

New CHB12 columns (24): ['alpha', 'beta', 'dataset', 'delta', 'delta_beta_ratio', 'entropy', 'epoch_idx', 'next_seizure_start_sec', 'recording_id', 'rel_alpha', 'rel_beta', 'rel_delta', 'rel_theta', 'rms', 'std', 'subject_id', 't_center_sec', 't_end_sec', 't_start_sec', 'theta', 'theta_alpha_ratio', 'time_to_next_seizure_sec', 'var', 'y_seizure']

qEEG feature columns identical? True

Columns only in existing CHB24 table: []
Columns only in new CHB12 table     : []

dtype comparison for shared columns:
  alpha                    existing=float64    new=float64    compatible=True


  theta_alpha_ratio        existing=float64    new=float64    compatible=True
  time_to_next_seizure_sec existing=float64    new=float64    compatible=True
  var                      existing=float64    new=float64    compatible=True
  y_seizure                existing=int64      new=int64      compatible=True

Missing values in new table (excluding intentional NaN forecasting cols): 0
Infinite values in new table: 0

[OK] CHB12 table is schema- and dtype-IDENTICAL to the existing CHB24 table


In [14]:
print("\n" + "=" * 60)
print("STEP 12: FEATURE SANITY (count/mean/std/min/max/NaN/Inf per feature)")
print("=" * 60)

sanity_rows = []
for col in qeeg_feature_cols:
    s = feat_chb12_all[col]
    sanity_rows.append({
        "feature": col,
        "count": s.count(),
        "mean": s.mean(),
        "std": s.std(),
        "min": s.min(),
        "max": s.max(),
        "n_nan": int(s.isna().sum()),
        "n_inf": int(np.isinf(s).sum()),
    })

sanity_df = pd.DataFrame(sanity_rows)
print(sanity_df.to_string(index=False))

corrupted = sanity_df[(sanity_df["n_nan"] > 0) | (sanity_df["n_inf"] > 0)]
if len(corrupted) > 0:
    raise ValueError(f"Corrupted qEEG features found:\n{corrupted}")

band_cols = ["delta", "theta", "alpha", "beta"]
rel_cols = ["rel_delta", "rel_theta", "rel_alpha", "rel_beta"]
nonneg_cols = ["std", "var", "rms"] + band_cols

for col in nonneg_cols:
    assert (feat_chb12_all[col] >= 0).all(), f"{col}: negative values found"
for col in rel_cols:
    assert (feat_chb12_all[col] >= -1e-9).all() and (feat_chb12_all[col] <= 1 + 1e-9).all(), \
        f"{col}: outside [0,1]"
assert np.isfinite(feat_chb12_all["entropy"]).all(), "entropy: non-finite values"

print("\n[OK] No NaN/Inf in qEEG features; band powers >= 0; relative powers in [0,1]")


STEP 12: FEATURE SANITY (count/mean/std/min/max/NaN/Inf per feature)


          feature  count         mean          std          min          max  n_nan  n_inf
              std    600 4.132626e-05 1.405171e-05 2.340177e-05 1.167079e-04      0      0
              var    600 1.904981e-09 1.485829e-09 5.476428e-10 1.362074e-08      0      0
              rms    600 4.132641e-05 1.405173e-05 2.340180e-05 1.167079e-04      0      0
            delta    600 1.213682e-09 1.037970e-09 2.698347e-10 1.043406e-08      0      0
            theta    600 2.234487e-10 2.062619e-10 6.176322e-11 2.281286e-09      0      0
            alpha    600 6.904617e-11 5.730852e-11 2.226668e-11 8.431820e-10      0      0
             beta    600 1.319832e-10 1.328733e-10 2.877356e-11 9.477234e-10      0      0
        rel_delta    600 7.126696e-01 1.081764e-01 2.656789e-01 9.278679e-01      0      0
        rel_theta    600 1.459075e-01 7.099847e-02 3.967279e-02 5.562564e-01      0      0
        rel_alpha    600 5.138148e-02 2.887806e-02 7.136088e-03 1.820379e-01      0      0

In [15]:
print("\n" + "=" * 60)
print("STEP 13: CHRONOLOGY VALIDATION")
print("=" * 60)

for recording_id, group in feat_chb12_all.groupby("recording_id"):
    idx = group["epoch_idx"].to_numpy()
    t_start = group["t_start_sec"].to_numpy()
    t_end = group["t_end_sec"].to_numpy()

    assert np.all(np.diff(idx) == 1), f"{recording_id}: epoch_idx not monotonic +1"
    assert np.all(np.diff(t_start) > 0), f"{recording_id}: t_start_sec not strictly increasing"
    assert np.all(t_end > t_start), f"{recording_id}: t_end_sec <= t_start_sec found"

    n_epochs = len(group)
    max_t_end = t_end.max()
    print(f"  {recording_id:14s}: {n_epochs:4d} epochs, t_start=[0, {t_start.max():.0f}]s, "
          f"max t_end={max_t_end:.0f}s")

print("\n[OK] epoch_idx monotonic (+1), t_start_sec strictly increasing, t_end_sec > t_start_sec "
      "for every recording")

print("\nRecording-boundary check (epoch t_end vs actual EDF duration):")
for rec_id in chb12_recording_ids:
    recording_id = f"chb_s{SUBJECT_ID:02d}_r{rec_id.split('_')[1]}"
    group = feat_chb12_all[feat_chb12_all["recording_id"] == recording_id]
    max_t_end = group["t_end_sec"].max()
    actual_duration = edf_metadata[rec_id]["duration_sec"]
    ok = max_t_end <= actual_duration
    print(f"  {recording_id:14s}: max_t_end={max_t_end:7.1f}s <= edf_duration={actual_duration:7.1f}s -> {ok}")
    assert ok, f"{recording_id}: epoch t_end exceeds recording duration -- boundary crossed"

print("\n[OK] No epoch crosses its own EDF recording boundary")


STEP 13: CHRONOLOGY VALIDATION
  chb_s12_r06   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s12_r08   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s12_r23   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s12_r38   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s12_r42   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s

[OK] epoch_idx monotonic (+1), t_start_sec strictly increasing, t_end_sec > t_start_sec for every recording

Recording-boundary check (epoch t_end vs actual EDF duration):
  chb_s12_r38   : max_t_end= 3600.0s <= edf_duration= 3600.0s -> True
  chb_s12_r42   : max_t_end= 3600.0s <= edf_duration= 3600.0s -> True
  chb_s12_r08   : max_t_end= 3600.0s <= edf_duration= 3600.0s -> True
  chb_s12_r23   : max_t_end= 3600.0s <= edf_duration= 3600.0s -> True


  chb_s12_r06   : max_t_end= 3600.0s <= edf_duration= 3606.0s -> True

[OK] No epoch crosses its own EDF recording boundary


In [16]:
print("\n" + "=" * 60)
print("STEP 14: SEIZURE MAPPING VALIDATION")
print("=" * 60)

print("Labeling convention: epoch positive iff [t_start_sec, t_end_sec) intersects")
print("[seizure_start, seizure_end) -- i.e. epoch_start < seizure_end AND epoch_end > seizure_start.\n")

print(f"{'recording':14s} {'onset':>7s} {'end':>7s} {'epochs_before_onset':>20s} {'overlapping_epochs':>19s}")
for recording_id, intervals in sorted(chb12_seizure_intervals_final.items()):
    group = feat_chb12_all[feat_chb12_all["recording_id"] == recording_id]
    for onset, end in intervals:
        n_before = int((group["t_end_sec"] <= onset).sum())
        overlap_mask = (group["t_start_sec"] < end) & (group["t_end_sec"] > onset)
        n_overlap = int(overlap_mask.sum())
        print(f"{recording_id:14s} {onset:7.0f} {end:7.0f} {n_before:20d} {n_overlap:19d}")

example_recording_id, example_intervals = sorted(chb12_seizure_intervals_final.items())[0]
example_onset, example_end = example_intervals[0]
example_group = feat_chb12_all[feat_chb12_all["recording_id"] == example_recording_id]
onset_epoch_idx = int(example_onset // EPOCH_DURATION_SEC)

print(f"\nBoundary illustration ({example_recording_id}, onset={example_onset:.0f}s, "
      f"end={example_end:.0f}s):")
for offset in (-1, 0, 1):
    ei = onset_epoch_idx + offset
    row = example_group[example_group["epoch_idx"] == ei]
    if len(row) == 0:
        continue
    row = row.iloc[0]
    is_positive = bool(row["y_seizure"])
    print(f"  epoch {ei:4d}: [{row['t_start_sec']:.0f}, {row['t_end_sec']:.0f})s -> "
          f"y_seizure={int(row['y_seizure'])} "
          f"({'intersects' if is_positive else 'does not intersect'} [{example_onset:.0f}, {example_end:.0f}))")

print("\nVerifying every y_seizure=1 epoch genuinely intersects an annotated interval...")
n_checked = 0
for recording_id, intervals in chb12_seizure_intervals_final.items():
    pos = feat_chb12_all[(feat_chb12_all["recording_id"] == recording_id) &
                          (feat_chb12_all["y_seizure"] == 1)]
    for _, row in pos.iterrows():
        intersects_any = any(
            row["t_start_sec"] < end and row["t_end_sec"] > start
            for start, end in intervals
        )
        assert intersects_any, (
            f"{recording_id} epoch {row['epoch_idx']}: y_seizure=1 but does not "
            f"intersect any annotated interval {intervals}"
        )
        n_checked += 1
print(f"[OK] {n_checked} positive epochs all verified to genuinely intersect an annotated seizure interval")


STEP 14: SEIZURE MAPPING VALIDATION
Labeling convention: epoch positive iff [t_start_sec, t_end_sec) intersects
[seizure_start, seizure_end) -- i.e. epoch_start < seizure_end AND epoch_end > seizure_start.

recording        onset     end  epochs_before_onset  overlapping_epochs
chb_s12_r06       1665    1726                   55                   3
chb_s12_r06       3415    3447                  113                   2


chb_s12_r08       1426    1439                   47                   1
chb_s12_r08       1591    1614                   53                   1
chb_s12_r08       1957    1977                   65                   1
chb_s12_r08       2798    2824                   93                   2
chb_s12_r23        253     333                    8                   4
chb_s12_r23        425     522                   14                   4
chb_s12_r23        630     670                   21                   2
chb_s12_r38       1548    1573                   51                   2
chb_s12_r38       2798    2821                   93                   2
chb_s12_r38       2966    3009                   98                   3
chb_s12_r38       3146    3201                  104                   3
chb_s12_r38       3364    3410                  112                   2
chb_s12_r42        699     750                   23                   2
chb_s12_r42        945     973                   31             

[OK] 41 positive epochs all verified to genuinely intersect an annotated seizure interval


In [17]:
print("\n" + "=" * 60)
print("CHB12 DATA PREPARATION -- EXECUTION REPORT")
print("=" * 60)

n_edfs = len(chb12_recording_ids)
n_seizure_recordings = sum(1 for v in chb12_seizure_intervals_final.values() if len(v) > 0)

print(f'''
Montage screening (from chb12-summary.txt, before any download):
  Seizure-containing files declared total   : {len(seizure_files_all)}
  Montage-compatible (all)                  : {len(compatible_files)} {compatible_files}
  Excluded -- incompatible montage           : {len(excluded_files)} {excluded_files}
    seizure events excluded (montage)        : {n_seizures_excluded}
  Excluded -- time-limited, NOT downloaded   : {len(not_fetched_in_time)} {not_fetched_in_time}
    seizure events excluded (time limit)     : {n_seizures_time_excluded}
  ACTUALLY DOWNLOADED + PROCESSED            : {len(chb12_recording_ids)} {chb12_recording_ids}

Files:
  EDFs processed               : {n_edfs}
  Seizure-containing recordings: {n_seizure_recordings}
  Total seizure events         : {n_seizure_events_total}
  Total downloaded size        : {total_bytes / 1e6:.1f} MB

Preprocessing compatibility with existing CHB24 representation:
  epoch duration    : {EPOCH_DURATION_SEC}s (matches)
  epoch overlap     : {EPOCH_OVERLAP} (matches)
  sampling frequency: {expected_chb_sfreq} Hz (matches, metadata-derived)
  filter            : {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz (matches)
  channels          : {len(CANONICAL_EEG_CHANNELS)} (post channel-drop, exact multiset match verified)
  qEEG feature count: {len(qeeg_feature_cols)} (identical names/formulas, verbatim reuse)

Output:
  {output_parquet_path}
  rows       : {feat_chb12_all.shape[0]}
  columns    : {feat_chb12_all.shape[1]}
  recordings : {n_edfs}
  seizures   : {n_seizure_events_total}
  NaNs (excl. forecasting cols): {n_nan_new}
  Infs       : {n_inf_new}
''')

print("=" * 60)
print("READY FOR CHB24 + CHB12 COMBINED RETRAIN")
print("=" * 60)


CHB12 DATA PREPARATION -- EXECUTION REPORT

Montage screening (from chb12-summary.txt, before any download):
  Seizure-containing files declared total   : 13
  Montage-compatible (all)                  : 10 ['chb12_06.edf', 'chb12_08.edf', 'chb12_09.edf', 'chb12_10.edf', 'chb12_11.edf', 'chb12_23.edf', 'chb12_33.edf', 'chb12_36.edf', 'chb12_38.edf', 'chb12_42.edf']
  Excluded -- incompatible montage           : 3 ['chb12_27.edf', 'chb12_28.edf', 'chb12_29.edf']
    seizure events excluded (montage)        : 13
  Excluded -- time-limited, NOT downloaded   : 5 ['chb12_09', 'chb12_10', 'chb12_33', 'chb12_11', 'chb12_36']
    seizure events excluded (time limit)     : 8
  ACTUALLY DOWNLOADED + PROCESSED            : 5 ['chb12_38', 'chb12_42', 'chb12_08', 'chb12_23', 'chb12_06']

Files:
  EDFs processed               : 5
  Seizure-containing recordings: 5
  Total seizure events         : 19
  Total downloaded size        : 261.9 MB

Preprocessing compatibility with existing CHB24 represent